# Tabletop detection - augmentation ablation (Kaggle GPU)

Follow-up to the first notebook. `aug_strong` (colour + rotation 10 deg + translate .2 + mixup .1) lowered val mAP50-95
from 0.815 to 0.657. Hypothesis: rotating a box-annotated object makes the axis-aligned boxes looser, which hurts
mAP50-95, while colour jitter alone may help because the 6 sessions differ a lot in lighting and surface.

Here the factors are separated: `aug_color` (hsv only) and `aug_rot10` (rotation only), against a freshly re-run baseline
(same machine, same seeds, so the comparison is clean). Same data and splits as before: train = s01-s04, val = s05, **no test**.

Before running: enable Internet + GPU, add the private dataset `tabletop-dataset` as input, then **Save & Run All (Commit)**.

In [ ]:
!pip install -q ultralytics==8.4.174
import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable a GPU in the settings")

In [ ]:
import shutil
from pathlib import Path
import yaml

# Locate the uploaded dataset (read-only input) and work on a writable copy.
candidates = list(Path("/kaggle/input").glob("**/dataset/train/images"))
assert candidates, "Dataset not found under /kaggle/input: add tabletop-dataset as notebook input."
src = candidates[0].parent.parent
dst = Path("/kaggle/working/dataset")
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)

NAMES = ["airpod", "charger", "fork", "pen", "spoon", "usb_key"]
DATA = "/kaggle/working/dataset.yaml"
Path(DATA).write_text(yaml.safe_dump({"path": str(dst), "train": "train/images", "val": "val/images",
                                       "names": NAMES}, sort_keys=False))
for split in ("train", "val"):
    print(split, len(list((dst / split / "images").glob("*.jpg"))), "images")
assert not (dst / "test").exists(), "the test split must not be in this notebook"

## Experiments
All use YOLO11n, COCO initialisation, 30 epochs, 640 px, patience 10 (same as `baseline_yolo11n`).

In [ ]:
SEEDS = [0, 1, 2]

EXPERIMENTS = {
    "baseline_yolo11n": dict(model="yolo11n.pt", epochs=30),
    "aug_color":        dict(model="yolo11n.pt", epochs=30, hsv_s=0.9, hsv_v=0.6),   # colour jitter only
    "aug_rot10":        dict(model="yolo11n.pt", epochs=30, degrees=10),            # rotation only
}

COMMON = dict(data=DATA, imgsz=640, batch=16, patience=10, device=0, workers=2,
              plots=False, verbose=False, exist_ok=True, project="/kaggle/working/runs")

In [ ]:
import time
from ultralytics import YOLO

rows = []
for exp, cfg in EXPERIMENTS.items():
    for seed in SEEDS:
        t0 = time.time()
        args = {**COMMON, **cfg, "seed": seed, "name": f"{exp}_s{seed}"}
        model = YOLO(args.pop("model"))
        model.train(**args)

        best = YOLO(f"/kaggle/working/runs/{args['name']}/weights/best.pt")
        m = best.val(data=DATA, split="val", imgsz=args["imgsz"], device=0, plots=False, verbose=False)
        row = {"experiment": exp, "seed": seed, "mAP50": m.box.map50, "mAP50-95": m.box.map,
               "P": m.box.mp, "R": m.box.mr, "minutes": (time.time() - t0) / 60}
        for i, c in enumerate(m.box.ap_class_index):
            row[f"AP50-95_{NAMES[int(c)]}"] = m.box.ap[i]
            row[f"AP50_{NAMES[int(c)]}"] = m.box.ap50[i]
        rows.append(row)
        print(f"{exp} seed={seed}: mAP50={row['mAP50']:.3f} mAP50-95={row['mAP50-95']:.3f} ({row['minutes']:.1f} min)")

In [ ]:
import pandas as pd
df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/aug_all_runs.csv", index=False)

cols = ["mAP50", "mAP50-95", "P", "R"] + [f"AP50-95_{n}" for n in NAMES]
summary = df.groupby("experiment", sort=False)[cols].agg(["mean", "std"]).round(3)
summary.to_csv("/kaggle/working/aug_summary_mean_std.csv")
pd.set_option("display.width", 220)
print(summary[["mAP50", "mAP50-95"]])
summary

In [ ]:
# Package results: metrics + per-run curves, and the best weights of seed 0 only (small files).
import zipfile
with zipfile.ZipFile("/kaggle/working/results_aug.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["aug_all_runs.csv", "aug_summary_mean_std.csv"]:
        z.write(f"/kaggle/working/{f}", f)
    for p in Path("/kaggle/working/runs").glob("*/results.csv"):
        z.write(p, f"runs/{p.parent.name}/results.csv")
    for p in Path("/kaggle/working/runs").glob("*_s0/weights/best.pt"):
        z.write(p, f"weights/{p.parent.parent.name}_best.pt")
print("results_aug.zip ready in /kaggle/working (download it from the Output tab)")